# VQE（変分量子固有値ソルバー）

量子回路で基底状態を近似的に作り、基底エネルギー$E_0$を求める方法。量子コンピュータで状態を作ってエネルギーを測り、古典コンピュータでパラメータを最適化する。以下ではQiskitで回路を組み、状態ベクトルシミュレータで計算する。

例として横磁場イジングモデル（TFIM）を使う。模型の詳しい説明は [models/01_tfim/00_model.ipynb](../models/01_tfim/00_model.ipynb) を参照。以下では

\begin{equation}
H = H_{ZZ} + H_X,\quad H_{ZZ} = -J\sum_j Z_jZ_{j+1},\quad H_X = -h\sum_j X_j
\end{equation}

と分けて書く。

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# 厳密対角化の関数（src/qmb/operators.py, models.py, ed.py）
from qmb.operators import plus_state
from qmb.models import tfim_hamiltonian as hamiltonian, magnetization_x
from qmb.ed import ground_state, time_evolve_expectation

# VQE の関数（src/qmb/circuits.py）
#   tfim_op     : H を Pauli 演算子の和（SparsePauliOp）で表したもの
#   hva_circuit : Hamiltonian Variational Ansatz（local=True でゲートごとのパラメータ）
#   energy, vqe : エネルギー期待値 E(θ) と、層を1つずつ増やす VQE
# 実装を見たいときは、Jupyterで関数名の後に ?? を付けて実行する（例: hamiltonian??）
from qmb.circuits import tfim_op, hva_circuit, energy, vqe

### 4-1 VQEの考え方

- **変分原理**

  任意の規格化された状態$\ket{\psi}$について

  \begin{equation}
  E(\psi) = \bra{\psi}H\ket{\psi} \ge E_0
  \end{equation}

  が成り立ち、等号は$\ket{\psi}$が基底状態のときに限る。そこでパラメータ$\boldsymbol{\theta}$を持つ量子回路で状態$\ket{\psi(\boldsymbol{\theta})}$を作り、$E(\boldsymbol{\theta})$を最小化すれば$E_0$に近づく。これがVQEである。

- **アルゴリズムの流れ**

  1. 量子コンピュータで$\ket{\psi(\boldsymbol{\theta})}$を作り、$E(\boldsymbol{\theta})$を測定する
  2. 古典コンピュータの最適化手法で$\boldsymbol{\theta}$を更新する
  3. $E(\boldsymbol{\theta})$が収束するまで1, 2を繰り返す

  $E$の測定では、$Z_jZ_{j+1}$の項は$Z$基底で、$X_j$の項は$X$基底で測る。このように2種類の測定をすればすべての項が求まる。

- **アンザッツ（回路の形）**

  回路の形をどう選ぶかが重要である。今回はTrotter分解（[03_trotter.ipynb](03_trotter.ipynb)）の1ステップと同じ形で、回転角を自由なパラメータにしたもの（**Hamiltonian Variational Ansatz, HVA**）を使う。

  \begin{equation}
  \ket{\psi(\boldsymbol{\beta}, \boldsymbol{\gamma})} = \prod_{l=1}^{p} e^{-i\gamma_l H_X}e^{-i\beta_l H_{ZZ}}\ket{+\cdots+}
  \end{equation}

  $p$は層の数で、パラメータは$2p$個である。$\ket{+\cdots+}$は$J=0$のときの基底状態なので、そこから$H_{ZZ}$と$H_X$を交互にかけて本当の基底状態に近づけていくイメージである（QAOAと同じ構造）。

  ただし開放境界条件では端があるため、サイトごとに最適な回転角が異なる。そこで各ゲートに別々のパラメータを持たせたもの

  \begin{equation}
  \ket{\psi(\boldsymbol{\theta})} = \prod_{l=1}^{p}\left(\prod_j e^{ih\gamma_{l,j}X_j}\prod_j e^{iJ\beta_{l,j}Z_jZ_{j+1}}\right)\ket{+\cdots+}
  \end{equation}

  も試す（パラメータは$p(2n-1)$個）。前者を「一様なHVA」、後者を「局所的なHVA」と呼ぶことにする。

> 周期境界条件では一様なHVAでも$p=n/2$層で厳密な基底状態に到達できることが知られている（W. W. Ho and T. H. Hsieh, SciPost Phys. 6, 029 (2019)）。

### 4-2 具体例を考えてみる

厳密対角化のノートブックの1-3と同じく、$n=2$（開放境界）の場合を手で計算する。

- **有効1スピンへの帰着**

  厳密対角化のノートブックの1-3で見たように、$\ket{++}$から出発すると状態は$\{\ket{\phi_+}, \ket{\psi_+}\}$の2次元部分空間から出ない。この部分空間で

  $$
  Z_1Z_2\ket{\phi_+} = \ket{\phi_+},\quad Z_1Z_2\ket{\psi_+} = -\ket{\psi_+},\quad (X_1+X_2)\ket{\phi_+} = 2\ket{\psi_+}
  $$

  なので、$\ket{\phi_+}$を$\ket{0}$、$\ket{\psi_+}$を$\ket{1}$とみなした1つの有効スピンのPauli行列$\sigma$を使うと

  $$
  H_{ZZ} = -J\sigma_z,\quad H_X = -2h\sigma_x,\quad M_x = \sigma_x,\quad \ket{++} = \frac{\ket{\phi_+}+\ket{\psi_+}}{\sqrt{2}} = \ket{+}_{\mathrm{eff}}
  $$

  と書ける。つまり$n=2$の問題は、磁場$(2h, 0, J)$中の1スピンの問題と同じである。

- **VQEによる基底エネルギー**

  一様なHVAで$p=1$とする。$e^{-i\beta H_{ZZ}} = e^{i\beta J\sigma_z}$は有効スピンを$z$軸まわりに$-2J\beta$回転させ、$e^{-i\gamma H_X} = e^{2i\gamma h\sigma_x}$は$x$軸まわりに$-4h\gamma$回転させる。$x$方向を向いた初期状態のBloch ベクトル$(1, 0, 0)$は

  $$
  (1,0,0) \to (\cos 2J\beta,\ -\sin 2J\beta,\ 0) \to (\cos 2J\beta,\ -\sin 2J\beta\cos 4h\gamma,\ \sin 2J\beta\sin 4h\gamma)
  $$

  と移る。エネルギーは$E = \langle H_{ZZ} + H_X\rangle = -J\langle\sigma_z\rangle - 2h\langle\sigma_x\rangle$なので

  $$
  E(\beta, \gamma) = -J\sin(2J\beta)\sin(4h\gamma) - 2h\cos(2J\beta)
  $$

  である。まず$\sin(4h\gamma) = 1$、すなわち$\gamma = \frac{\pi}{8h}$とすると$E = -J\sin(2J\beta) - 2h\cos(2J\beta)$となる。これを$\beta$について最小化すると、$\tan(2J\beta) = \frac{J}{2h}$のとき

  $$
  E_{\min} = -\sqrt{J^2+4h^2} = E_0
  $$

  となり、$n=2$では$p=1$で厳密な基底エネルギーが得られる。$J=1, h=0.5$では最適なパラメータは$\beta = \frac{\pi}{8},\ \gamma = \frac{\pi}{4}$である。

### 4-3 コードで確かめる

Qiskitで回路を組んで計算する。回路を作る関数は `src/qmb/circuits.py` にまとめてあり、ノートブック冒頭で読み込んでいる。Qiskitは量子ビットの並びが厳密対角化で使った`np.kron`と逆順だが、今回扱う$H$と$M_x$は左右反転で変わらないので、そのまま比較できる。

In [2]:
# 例: N=2, J=1, h=0.5, p=1
N = 2
J = 1.0
h = 0.5
E_vqe, params = vqe(N, J, h, p_max=1)
qc = hva_circuit(N, J, h, p=1)
print(qc.draw())
print(f"VQE で得たパラメータ   β = {params[0]:.6f}, γ = {params[1]:.6f}")
print(f"手計算の最適パラメータ β = π/8 = {np.pi / 8:.6f}, γ = π/4 = {np.pi / 4:.6f}")
print(f"E(π/8, π/4)       = {energy(qc, tfim_op(N, J, h), [np.pi / 8, np.pi / 4]):.6f}")
print(f"基底エネルギー E0 (VQE)  = {E_vqe[-1]:.6f}")
print(f"基底エネルギー E0 (厳密) = {-np.sqrt(J**2 + 4 * h**2):.6f}")

     ┌───┐                ┌───────────┐
q_0: ┤ H ├─■──────────────┤ Rx(-θ[1]) ├
     ├───┤ │ZZ((-2)*θ[0]) ├───────────┤
q_1: ┤ H ├─■──────────────┤ Rx(-θ[1]) ├
     └───┘                └───────────┘
VQE で得たパラメータ   β = 0.392699, γ = 0.785398
手計算の最適パラメータ β = π/8 = 0.392699, γ = π/4 = 0.785398
E(π/8, π/4)       = -1.414214
基底エネルギー E0 (VQE)  = -1.414214
基底エネルギー E0 (厳密) = -1.414214


$p=1$で厳密な基底エネルギーが得られ、最適なパラメータも手計算と一致した。

$n$を大きくしたときに層の数$p$がいくつ必要か（一様なHVAと局所的なHVAの比較）は、[models/01_tfim/01_ground_state.ipynb](../models/01_tfim/01_ground_state.ipynb)で調べる。

---
**前へ**: [03 Trotter分解](03_trotter.ipynb)　｜　**次へ**: [05 TEBD](05_tebd.ipynb)